In [ ]:
# import shutil
# from google.colab import drive
# drive.mount('/content/drive')

# shutil.unpack_archive('/content/drive/MyDrive/cnn_dataset.zip', '/content')
# print("cnn_dataset 압축 해제 완료")


In [2]:
# =====================================================================
# 3단계: CNN 분류기 학습 (MobileNetV2 전이학습, freeze / 파인튜닝 두 버전)
# =====================================================================
# 2_crop_bboxes_for_cnn.py 로 만든 cnn_dataset/ 폴더를 사용합니다.
# cnn_dataset/train/plastic, cnn_dataset/train/glass, cnn_dataset/train/can ...
#
# freeze 버전과 파인튜닝 버전을 각각 학습해서 검증 정확도를 비교한 뒤,
# 더 나은 쪽을 최종 모델로 선택하면 됩니다. (Google Colab GPU 권장)
# =====================================================================

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"  # TF 2.16+ 기본 Keras 3와 TFLiteConverter 호환성 버그 회피

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")  # RandomBrightness가 tensorflow-metal(GPU)에서 커널 크래시를 일으켜 CPU로 강제
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
DATA_DIR = "cnn_dataset"   # 2번 스크립트의 OUTPUT_DIR과 동일 경로로 수정

# --- 데이터 불러오기 ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/valid", image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("클래스:", class_names)  # 예: ['can', 'glass', 'plastic']

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 데이터 증강: 위치가 살짝씩 달라지는 실제 환경 대비
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),  # 조명 변화 대비 (유리·캔 반사 이슈 보완)
])
normalization = layers.Rescaling(1.0 / 127.5, offset=-1)  # MobileNetV2 표준 전처리


def build_model(base_trainable=False, fine_tune_at=100):
    base = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
    )
    base.trainable = base_trainable
    if base_trainable:
        # fine_tune_at 이전 레이어는 계속 고정, 이후 레이어만 재학습
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = normalization(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)

    model = models.Model(inputs, outputs)
    lr = 1e-5 if base_trainable else 1e-3  # 파인튜닝은 학습률을 훨씬 낮게
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# --- 1) freeze 버전: 특징 추출부 고정, 마지막 분류층만 학습 ---
print("\n=== freeze 버전 학습 ===")
freeze_model = build_model(base_trainable=False)
freeze_history = freeze_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2
)
freeze_model.save("model_freeze.keras")

# --- 2) 파인튜닝 버전: 상위 레이어까지 낮은 학습률로 재학습 ---
print("\n=== 파인튜닝 버전 학습 ===")
finetune_model = build_model(base_trainable=True, fine_tune_at=100)
finetune_history = finetune_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2
)
finetune_model.save("model_finetune.keras")

# --- 3) 두 모델 검증 정확도 비교 ---
freeze_loss, freeze_acc = freeze_model.evaluate(val_ds, verbose=0)
finetune_loss, finetune_acc = finetune_model.evaluate(val_ds, verbose=0)

best_model = finetune_model if finetune_acc >= freeze_acc else freeze_model
best_name = "finetune" if finetune_acc >= freeze_acc else "freeze"

print("\n" + "=" * 50)
print("최종 결과 요약")
print("=" * 50)
print(f"클래스: {class_names}")
print(f"freeze   검증 정확도: {freeze_acc * 100:6.2f}%   (loss {freeze_loss:.4f})")
print(f"finetune 검증 정확도: {finetune_acc * 100:6.2f}%   (loss {finetune_loss:.4f})")
print(f"-> 최종 선택 모델: {best_name} 버전")
print("=" * 50)

# --- 4) 최종 모델을 .tflite로 변환 (Jetson Nano 등 경량 기기 배포용) ---
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # 크기/속도 최적화
tflite_model = converter.convert()

with open("classifier.tflite", "wb") as f:
    f.write(tflite_model)

# 클래스 순서도 같이 저장해둬야 나중에 추론 결과 인덱스를 해석할 수 있음
with open("class_names.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(class_names))

print("\n완료: classifier.tflite, class_names.txt 저장됨")

# =====================================================================
# 다음 단계
# 1) best.pt (YOLO), classifier.tflite (CNN), class_names.txt 를
#    Jetson Nano로 옮김
# 2) 실제 카메라로 YOLO 검출 -> 바운딩박스 크롭 -> CNN 추론 파이프라인 테스트
# 3) 정지-촬영 방식이면: IR센서 감지 -> 벨트 정지 -> 촬영 -> 이 파이프라인
#    -> 서보 매핑 -> 벨트 재가동 순서로 통합
# =====================================================================


Found 2724 files belonging to 3 classes.
Found 747 files belonging to 3 classes.
클래스: ['metal', 'paper', 'plastic']

=== freeze 버전 학습 ===


Epoch 1/15
86/86 - 28s - loss: 0.6429 - accuracy: 0.7331 - val_loss: 0.3960 - val_accuracy: 0.8380 - 28s/epoch - 325ms/step
Epoch 2/15
86/86 - 21s - loss: 0.3914 - accuracy: 0.8502 - val_loss: 0.3384 - val_accuracy: 0.8795 - 21s/epoch - 246ms/step
Epoch 3/15
86/86 - 21s - loss: 0.3458 - accuracy: 0.8605 - val_loss: 0.3168 - val_accuracy: 0.8942 - 21s/epoch - 244ms/step
Epoch 4/15
86/86 - 22s - loss: 0.3080 - accuracy: 0.8800 - val_loss: 0.2919 - val_accuracy: 0.8996 - 22s/epoch - 252ms/step
Epoch 5/15
86/86 - 21s - loss: 0.2874 - accuracy: 0.8833 - val_loss: 0.2779 - val_accuracy: 0.9009 - 21s/epoch - 244ms/step
Epoch 6/15
86/86 - 24s - loss: 0.2753 - accuracy: 0.8950 - val_loss: 0.2777 - val_accuracy: 0.9023 - 24s/epoch - 277ms/step
Epoch 7/15
86/86 - 22s - loss: 0.2696 - accuracy: 0.8961 - val_loss: 0.2731 - val_accuracy: 0.9009 - 22s/epoch - 260ms/step
Epoch 8/15
86/86 - 24s - loss: 0.2575 - accuracy: 0.8961 - val_loss: 0.2828 - val_accuracy: 0.9009 - 24s/epoch - 274ms/step
Epoch 9/

Epoch 1/15
86/86 - 34s - loss: 0.7817 - accuracy: 0.6446 - val_loss: 0.4494 - val_accuracy: 0.8112 - 34s/epoch - 391ms/step
Epoch 2/15
86/86 - 25s - loss: 0.4199 - accuracy: 0.8359 - val_loss: 0.3685 - val_accuracy: 0.8514 - 25s/epoch - 293ms/step
Epoch 3/15
86/86 - 26s - loss: 0.3419 - accuracy: 0.8697 - val_loss: 0.3095 - val_accuracy: 0.8675 - 26s/epoch - 300ms/step
Epoch 4/15
86/86 - 25s - loss: 0.2870 - accuracy: 0.8899 - val_loss: 0.2844 - val_accuracy: 0.8768 - 25s/epoch - 290ms/step
Epoch 5/15
86/86 - 25s - loss: 0.2560 - accuracy: 0.9027 - val_loss: 0.2587 - val_accuracy: 0.8929 - 25s/epoch - 291ms/step
Epoch 6/15
86/86 - 24s - loss: 0.2398 - accuracy: 0.9035 - val_loss: 0.2568 - val_accuracy: 0.8876 - 24s/epoch - 285ms/step
Epoch 7/15
86/86 - 25s - loss: 0.2293 - accuracy: 0.9082 - val_loss: 0.2313 - val_accuracy: 0.9050 - 25s/epoch - 295ms/step
Epoch 8/15
86/86 - 25s - loss: 0.2168 - accuracy: 0.9156 - val_loss: 0.2265 - val_accuracy: 0.9090 - 25s/epoch - 289ms/step
Epoch 9/

INFO:tensorflow:Assets written to: /var/folders/jt/jnmwv9t15h525db1wzbrn8580000gn/T/tmpv_yl5p4a/assets



완료: classifier.tflite, class_names.txt 저장됨


W0000 00:00:1785176539.045590 9506596 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1785176539.047694 9506596 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
2026-07-28 03:22:19.050316: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jt/jnmwv9t15h525db1wzbrn8580000gn/T/tmpv_yl5p4a
2026-07-28 03:22:19.065679: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-28 03:22:19.065697: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jt/jnmwv9t15h525db1wzbrn8580000gn/T/tmpv_yl5p4a
2026-07-28 03:22:19.186414: I tensorflow/cc/saved_model/loader.cc:234] Restoring SavedModel bundle.
2026-07-28 03:22:19.512415: I tensorflow/cc/saved_model/loader.cc:218] Running initialization op on SavedModel bundle at path: /var/folders/jt/jnmwv9t15h525db1wzbrn8580000gn/T/tmpv_yl5p4a
2026-07-28 03:22:19.615477: I tensorflow/cc/saved_model/loader.cc: